# YOLO11m-cls — 17 Species Classifier (Colab)

Uses local files (synced by VSCode Colab extension), computes on Colab's GPU.
Dataset is at `../datasets/final_17species/` relative to this notebook.

In [ ]:
# ── Setup ──────────────────────────────────────────────────
import shutil, random
from pathlib import Path

import cv2
import numpy as np
from tqdm import tqdm

from ultralytics import YOLO, settings

In [ ]:
# ── Paths (local, synced by Colab extension) ───────────────

NOTEBOOK_DIR = Path.cwd()                       # notebooks/
PROJECT_ROOT = NOTEBOOK_DIR.parent              # Vision/
DATA = PROJECT_ROOT / "datasets" / "final_17species"
CACHE = PROJECT_ROOT / ".cache" / "cache_17species.mmap"
MODELS = PROJECT_ROOT / "models" / "yolo"
RUNS = MODELS / "runs"

# ── Training Config ────────────────────────────────────────

MODEL_NAME = "yolo11m-cls.pt"
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 200
PATIENCE = 30
SEED = 42
DEVICE = 0                          # Colab GPU (T4)
KEEP_LAST_N = 15                    # keep epoch_N.pt for last N epochs only

# Augmentation
HSV_H, HSV_S, HSV_V = 0.05, 0.80, 0.50
DEGREES, TRANSLATE, SCALE, SHEAR = 15.0, 0.15, 0.35, 8.0
FLIPLR, FLIPUD, ERASING = 0.5, 0.1, 0.45

# LR
LR0, LRF = 0.001, 0.00001
WARMUP_EPOCHS, LABEL_SMOOTHING = 5, 0.1
MOMENTUM, WEIGHT_DECAY = 0.937, 5e-4

random.seed(SEED)

In [ ]:
# ── Sanity check ───────────────────────────────────────────
if not DATA.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA}")

n_species = len(list((DATA / "train").iterdir()))
n_train = sum(1 for _ in (DATA / "train").rglob("*.jpg"))
n_val = sum(1 for _ in (DATA / "val").rglob("*.jpg"))
n_test = sum(1 for _ in (DATA / "test").rglob("*.jpg")) if (DATA / "test").exists() else 0

print(f"  Dataset: {DATA}")
print(f"  {n_species} species, {n_train:,} train / {n_val:,} val / {n_test:,} test")
print(f"  Runs go to: {RUNS}")

---
## Step 1 — Create .mmap cache (one-time)

In [ ]:
if CACHE.exists():
    print(f"  ✅ Using existing cache: {CACHE.name} ({CACHE.stat().st_size/1024**3:.1f} GB)")
else:
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    images = sorted((DATA / "train").rglob("*.jpg"))
    n, h, w = len(images), IMG_SIZE, IMG_SIZE

    print(f"  Caching {n} images to {CACHE.name}...")
    mm = np.memmap(str(CACHE), dtype=np.uint8, mode="w+", shape=(n, h, w, 3))
    failed = 0
    for i, path in enumerate(tqdm(images, desc="  Caching", unit="img")):
        img = cv2.imread(str(path))
        if img is None:
            failed += 1
            continue
        mm[i] = cv2.resize(img, (w, h))
    mm.flush()
    print(f"  ✅ {CACHE.name} ({CACHE.stat().st_size/1024**3:.1f} GB, {failed} failed)")
    del mm

---
## Step 2 — Train (with tail-end checkpoint saving)

In [ ]:
# ── Callback: keep last N epoch checkpoints ────────────────
def keep_last_n_epochs(trainer):
    """After each epoch in the tail window, save epoch_N.pt.
    Delete epoch files older than the window."""
    epoch = trainer.epoch
    weights_dir = Path(trainer.save_dir) / "weights"
    keep_start = max(0, trainer.epochs - KEEP_LAST_N)

    if epoch >= keep_start:
        src = weights_dir / "last.pt"
        dst = weights_dir / f"epoch{epoch}.pt"
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            print(f"    📦 Saved {dst.name}")

    for f in sorted(weights_dir.glob("epoch*.pt")):
        ep = int(f.stem.replace("epoch", ""))
        if ep < keep_start:
            f.unlink()


# ── Point YOLO settings ────────────────────────────────────
settings.update({
    "runs_dir": str(RUNS),
    "weights_dir": str(MODELS),
})

# ── Train ──────────────────────────────────────────────────
print(f"  {MODEL_NAME}, {EPOCHS} epochs, batch {BATCH_SIZE}, device={DEVICE}")

model = YOLO(MODEL_NAME)
model.add_callback("on_fit_epoch_end", keep_last_n_epochs)

model.train(
    data=str(DATA),
    imgsz=IMG_SIZE, batch=BATCH_SIZE, epochs=EPOCHS,
    patience=PATIENCE, device=DEVICE, workers=0, seed=SEED,
    deterministic=False,
    hsv_h=HSV_H, hsv_s=HSV_S, hsv_v=HSV_V,
    degrees=DEGREES, translate=TRANSLATE, scale=SCALE, shear=SHEAR,
    fliplr=FLIPLR, flipud=FLIPUD, erasing=ERASING,
    lr0=LR0, lrf=LRF, warmup_epochs=WARMUP_EPOCHS, cos_lr=True,
    momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, optimizer="SGD",
    label_smoothing=LABEL_SMOOTHING, dropout=0.15,
    save=True, save_period=EPOCHS + 1,  # disable periodic saves; callback handles tail
    project=str(RUNS), name="species_yolo11m", exist_ok=True,
    val=True, plots=True,
)

print(f"  ✅ Training done")

---
## Done

**All outputs are local** (synced back from Colab runtime):
- `models/yolo/runs/species_yolo11m/weights/` — `best.pt`, `last.pt`, `epoch_{N}.pt` (last 15)
- `models/yolo/runs/species_yolo11m/` — plots, confusion matrix, results CSV

No Drive mount, no Kaggle download. Everything lives in your project folder.